In [1805]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

import re

font_path = 'C:/Windows/Fonts/gulim.ttc'
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font",family=font)

In [1806]:
df_team = pd.read_csv("../../../EDA/Merge/data/result data/Final DF.csv")  # Year, Nation, ..., DEF_INDEX 등

# 예시: 경기 단위 데이터
df_match = pd.read_csv("./../data/matches_1930_2022.csv")  

In [1807]:
df_team.columns
df_team=df_team[['Year', 'Nation','Eng_Nation','Q_WR', 'Q_GR',
       'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age','Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5', 'FS_6',
       'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13']]

In [1808]:
df_match=df_match[['Year','home_team','away_team','home_score','away_score','Score']]
df_fin = df_match

In [1809]:
country_map = {
    "브라질": "Brazil",
    "독일": "Germany",
    "터키": "Türkiye",
    "대한민국": "Korea Republic",
    "스페인": "Spain",
    "잉글랜드": "England",
    "세네갈": "Senegal",
    "미국": "United States",
    "일본": "Japan",
    "덴마크": "Denmark",
    "멕시코": "Mexico",
    "아일랜드": "Republic of Ireland",
    "스웨덴": "Sweden",
    "벨기에": "Belgium",
    "이탈리아": "Italy",
    "파라과이": "Paraguay",
    "남아프리카 공화국": "South Africa",
    "아르헨티나": "Argentina",
    "코스타리카": "Costa Rica",
    "카메룬": "Cameroon",
    "포르투갈": "Portugal",
    "러시아": "Russia",
    "크로아티아": "Croatia",
    "에콰도르": "Ecuador",
    "폴란드": "Poland",
    "우루과이": "Uruguay",
    "나이지리아": "Nigeria",
    "프랑스": "France",
    "튀니지": "Tunisia",
    "슬로베니아": "Slovenia",
    "중국": "China PR",
    "사우디아라비아": "Saudi Arabia",
    "우크라이나": "Ukraine",
    "스위스": "Switzerland",
    "네덜란드": "Netherlands",
    "가나": "Ghana",
    "호주": "Australia",
    "코트디부아르": "Côte d'Ivoire",
    "체코": "Czech Republic",
    "앙골라": "Angola",
    "이란": "IR Iran",
    "트리니다드 토바고": "Trinidad and Tobago",
    "토고": "Togo",
    "세르비아 몬테네그로": "Serbia and Montenegro",
    "칠레": "Chile",
    "슬로바키아": "Slovakia",
    "뉴질랜드": "New Zealand",
    "세르비아": "Serbia",
    "그리스": "Greece",
    "알제리": "Algeria",
    "온두라스": "Honduras",
    "북한": "Korea DPR",
    "콜롬비아": "Colombia",
    "보스니아 헤르체고비나": "Bosnia and Herzegovina",
    "페루": "Peru",
    "모로코": "Morocco",
    "아이슬란드": "Iceland",
    "이집트": "Egypt",
    "파나마": "Panama",
    "웨일스": "Wales",
    "캐나다": "Canada",
    "카타르": "Qatar",
}

In [1810]:


# ---------------------------
# 0️⃣ 2002년 이후 경기만 선택
# ---------------------------
df_match_filtered = df_match[df_match['Year'] >= 2002].reset_index(drop=True)

# ---------------------------
# 1️⃣ 팀 데이터에 영어 이름 컬럼 추가 (country_map 사용)
# ---------------------------
df_team['Eng_Nation'] = df_team['Nation'].map(country_map)

# ---------------------------
# 2️⃣ Year + Eng_Nation 기준 인덱스 설정
# ---------------------------
df_team_renamed = df_team.set_index(['Year', 'Eng_Nation'])

# ---------------------------
# 3️⃣ 매핑 실패 팀 기록용
# ---------------------------
missing_teams = set()

# ---------------------------
# 4️⃣ 점수 문자열에서 최종 결과 계산
# ---------------------------
def parse_result_from_score(score_str):
    """
    score_str 예시: "3–3", "(4) 3–3 (2)", "2–1"
    반환: 1 = 홈승, 0 = 무승부, -1 = 홈패
    """
    # 정규/연장 점수 추출
    main_match = re.search(r'(\d+)\s*–\s*(\d+)', score_str)
    if not main_match:
        return None  # 점수 형식 이상
    
    home_main = int(main_match.group(1))
    away_main = int(main_match.group(2))
    
    # 점수가 다르면 정규/연장으로 승패 결정
    if home_main > away_main:
        return 1
    elif home_main < away_main:
        return -1
    else:
        # 동점이면 승부차기 점수 확인
        pens = re.findall(r'\((\d+)\)', score_str)
        if len(pens) == 2:
            home_pen = int(pens[0])
            away_pen = int(pens[1])
            if home_pen > away_pen:
                return 1
            elif home_pen < away_pen:
                return -1
        # 승부차기 없거나 동점이면 무승부
        return 0

# ---------------------------
# 5️⃣ 홈/어웨이 팀 특징 매핑 함수
# ---------------------------
def map_team_features_safe(row):
    year = row['Year']
    home = row['home_team']
    away = row['away_team']
    
    # 홈팀 특징
    try:
        home_feat = df_team_renamed.loc[(year, home)].add_prefix("Home_")
    except KeyError:
        missing_teams.add(home)
        home_feat = pd.Series(dtype=float)
    
    # 어웨이팀 특징
    try:
        away_feat = df_team_renamed.loc[(year, away)].add_prefix("Away_")
    except KeyError:
        missing_teams.add(away)
        away_feat = pd.Series(dtype=float)
    
    # 경기 결과 계산 (홈승=1, 무승부=0, 홈패=-1)
    result = parse_result_from_score(row['Score'])

    return pd.concat([pd.Series({'Year': year}), home_feat, away_feat, pd.Series({'Result': result})])

# ---------------------------
# 6️⃣ 적용
# ---------------------------
df_final = df_match_filtered.apply(map_team_features_safe, axis=1).reset_index(drop=True)

# ---------------------------
# 7️⃣ 매핑 실패 팀 확인
# ---------------------------
if missing_teams:
    print("매핑이 안 된 팀:", missing_teams)

# ---------------------------
# 8️⃣ 결과 확인
# ---------------------------
print(df_final.head())


   Year Home_Nation  Home_Q_WR  Home_Q_GR  Home_F_Rank  Home_F_Point  \
0  2022       아르헨티나       0.52   3.888889         8.33       1654.10   
1  2022       크로아티아       0.54   1.479452        11.00       1620.42   
2  2022         프랑스       0.63   2.830189         2.67       1741.10   
3  2022       아르헨티나       0.52   3.888889         8.33       1654.10   
4  2022         모로코       0.51   3.055556        38.33       1470.74   

   Home_F_Rd  Home_F_Pd  Home_Avg_Apps  Home_Avg_Age  ...  Away_FS_5  \
0       6.58     190.35        0.50000         27.81  ...      56.58   
1      -4.00     592.92        0.46154         27.42  ...      54.26   
2     -10.66     644.77        0.65385         26.69  ...      54.26   
3       6.58     190.35        0.50000         27.81  ...      59.85   
4     -28.00     938.74        0.23077         26.31  ...      59.48   

   Away_FS_6  Away_FS_7  Away_FS_8  Away_FS_9  Away_FS_10  Away_FS_11  \
0      71.88      68.46      74.35      68.04       73.38    

In [1811]:
df = df_final
df.columns

Index(['Year', 'Home_Nation', 'Home_Q_WR', 'Home_Q_GR', 'Home_F_Rank',
       'Home_F_Point', 'Home_F_Rd', 'Home_F_Pd', 'Home_Avg_Apps',
       'Home_Avg_Age', 'Home_Avg_Famous', 'Home_FS_0', 'Home_FS_1',
       'Home_FS_2', 'Home_FS_3', 'Home_FS_4', 'Home_FS_5', 'Home_FS_6',
       'Home_FS_7', 'Home_FS_8', 'Home_FS_9', 'Home_FS_10', 'Home_FS_11',
       'Home_FS_12', 'Home_FS_13', 'Away_Nation', 'Away_Q_WR', 'Away_Q_GR',
       'Away_F_Rank', 'Away_F_Point', 'Away_F_Rd', 'Away_F_Pd',
       'Away_Avg_Apps', 'Away_Avg_Age', 'Away_Avg_Famous', 'Away_FS_0',
       'Away_FS_1', 'Away_FS_2', 'Away_FS_3', 'Away_FS_4', 'Away_FS_5',
       'Away_FS_6', 'Away_FS_7', 'Away_FS_8', 'Away_FS_9', 'Away_FS_10',
       'Away_FS_11', 'Away_FS_12', 'Away_FS_13', 'Result'],
      dtype='object')

In [1812]:
# 홈/어웨이 컬럼 분리
home_cols = [col for col in df.columns if col.startswith('Home_')]
away_cols = [col for col in df.columns if col.startswith('Away_')]

# 컬럼 이름 접두사 바꾸기
home_rename = {col: col.replace('Home_', 'Away_') for col in home_cols}
away_rename = {col: col.replace('Away_', 'Home_') for col in away_cols}

# 데이터 복사 후 컬럼 교체
df_swapped = df.copy()
df_swapped = df_swapped.rename(columns={**home_rename, **away_rename})

# Result 반대로 바꾸기
df_swapped['Result'] = df_swapped['Result'].replace({1: -1, -1: 1, 0: 0})

# 원본과 합치기
df_expanded = pd.concat([df, df_swapped], ignore_index=True)

print(df_expanded.shape)
print(df_expanded.head())


(768, 50)
   Year Home_Nation  Home_Q_WR  Home_Q_GR  Home_F_Rank  Home_F_Point  \
0  2022       아르헨티나       0.52   3.888889         8.33       1654.10   
1  2022       크로아티아       0.54   1.479452        11.00       1620.42   
2  2022         프랑스       0.63   2.830189         2.67       1741.10   
3  2022       아르헨티나       0.52   3.888889         8.33       1654.10   
4  2022         모로코       0.51   3.055556        38.33       1470.74   

   Home_F_Rd  Home_F_Pd  Home_Avg_Apps  Home_Avg_Age  ...  Away_FS_5  \
0       6.58     190.35        0.50000         27.81  ...      56.58   
1      -4.00     592.92        0.46154         27.42  ...      54.26   
2     -10.66     644.77        0.65385         26.69  ...      54.26   
3       6.58     190.35        0.50000         27.81  ...      59.85   
4     -28.00     938.74        0.23077         26.31  ...      59.48   

   Away_FS_6  Away_FS_7  Away_FS_8  Away_FS_9  Away_FS_10  Away_FS_11  \
0      71.88      68.46      74.35      68.04      

In [1813]:
home_cols = [col for col in df_final.columns if col.startswith('Home_')]
away_cols = [col for col in df_final.columns if col.startswith('Away_')]

# Nation 같은 문자열 컬럼 제외
home_cols = [c for c in home_cols if c not in ["Home_Nation"]]
away_cols = [c for c in away_cols if c not in ["Away_Nation"]]

# 공통 지표 이름 추출 (Home_ 접두사 제거)
metrics = [col.replace("Home_", "") for col in home_cols]

diff_data = {}

for metric in metrics:
    home_col = f"Home_{metric}"
    away_col = f"Away_{metric}"
    
    # 두 컬럼이 존재하고 수치형일 때만 처리
    if home_col in df_final.columns and away_col in df_final.columns:
        if pd.api.types.is_numeric_dtype(df_final[home_col]) and pd.api.types.is_numeric_dtype(df_final[away_col]):
            diff_data[metric] = df_final[home_col] - df_final[away_col]

# 결과 컬럼 추가
diff_data["Result"] = df_final["Result"]
diff_data["Year"] = df_final["Year"]


# Home, Away 팀 한글 → 영어 매핑
diff_data["Home_Eng_Nation"] = df_final["Home_Nation"].map(country_map)
diff_data["Away_Eng_Nation"] = df_final["Away_Nation"].map(country_map)

# 최종 데이터프레임 생성
df_diff = pd.DataFrame(diff_data)

print(df_diff.head())


   Q_WR      Q_GR  F_Rank  F_Point   F_Rd    F_Pd  Avg_Apps  Avg_Age  \
0 -0.11  1.058700    5.66   -87.00  17.24 -454.42  -0.15385     1.12   
1  0.03 -1.576104  -27.33   149.68  24.00 -345.82   0.23077     1.11   
2  0.12 -0.225367  -35.66   270.36  17.34 -293.97   0.42308     0.38   
3 -0.02  2.409437   -2.67    33.68  10.58 -402.57   0.03846     0.39   
4 -0.16 -0.476359   32.66  -186.23 -28.17  528.27  -0.38461    -0.57   

   Avg_Famous  FS_0  ...   FS_8  FS_9  FS_10  FS_11  FS_12  FS_13  Result  \
0           2 -1.34  ...  -5.54  1.69  -1.96  -0.25  -1.94  -0.90       1   
1           1  0.76  ...   5.29  1.41   0.62  -5.60   3.38   7.11       1   
2           5  7.76  ...  10.26  2.95   5.73  -2.80   9.47  11.02       1   
3           6  5.66  ...  -0.57  3.23   3.15   2.55   4.15   3.01       1   
4          -9 -5.61  ...  -3.71 -6.35  -1.07   2.82  -5.97  -6.29       1   

   Year  Home_Eng_Nation  Away_Eng_Nation  
0  2022        Argentina           France  
1  2022         

In [1814]:
df_diff = df_diff[df_diff['Result'] != 0]
df_diff['Result'].unique()

array([ 1, -1])

In [1815]:
df_diff.to_csv('./../data/before_record.csv')

In [1816]:
df_record = pd.read_csv('./../data/results.csv')
df_record.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48532 entries, 0 to 48531
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        48532 non-null  object
 1   home_team   48532 non-null  object
 2   away_team   48532 non-null  object
 3   home_score  48532 non-null  int64 
 4   away_score  48532 non-null  int64 
 5   tournament  48532 non-null  object
 6   city        48532 non-null  object
 7   country     48532 non-null  object
 8   neutral     48532 non-null  bool  
dtypes: bool(1), int64(2), object(6)
memory usage: 3.0+ MB


In [1817]:
df_record = df_record[(df_record['date'] > '1990-01-01') & (df_record['date'] < '2023-01-01')]

In [1818]:
df_record.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28651 entries, 17113 to 45763
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        28651 non-null  object
 1   home_team   28651 non-null  object
 2   away_team   28651 non-null  object
 3   home_score  28651 non-null  int64 
 4   away_score  28651 non-null  int64 
 5   tournament  28651 non-null  object
 6   city        28651 non-null  object
 7   country     28651 non-null  object
 8   neutral     28651 non-null  bool  
dtypes: bool(1), int64(2), object(6)
memory usage: 2.0+ MB


In [1819]:
# 세부 리그 대략적인 큰 범위로 축소
def categorize_tournament(t):
    if t == "FIFA World Cup":
        return "World Cup "
    elif t == "FIFA World Cup qualification":
        return "World Cup Qualifiers"
    elif "Euro" in t or "UEFA Nations League" in t:
        return "Europe"
    elif "AFC" in t or "Asian" in t or "EAFF" in t or "SAFF" in t or "AFF" in t or "CAFA" in t or "ASEAN" in t:
        return "Asia"
    elif "Africa" in t or "COSAFA" in t or "CECAFA" in t or "UNIFFAC" in t or "UDEAC" in t:
        return "Africa"
    elif "Copa" in t or "Gold Cup" in t or "CONCACAF" in t or "CONMEBOL" in t:
        return "Americas"
    elif "Oceania" in t or "Pacific" in t:
        return "Oceania"
    elif "Friendly" in t:
        return "Friendly"
    else:
        return "Other/Regional"

df_record["tournament_category"] = df_record["tournament"].apply(categorize_tournament)

In [1820]:
df_record['tournament_category'].unique()
df_record.loc[df_record['tournament_category'] == 'World Cup ', 'tournament_category'] = 'World Cup Qualifiers'
df_record['tournament_category'].unique()

array(['Friendly', 'Other/Regional', 'Africa', 'Europe',
       'World Cup Qualifiers', 'Asia', 'Americas', 'Oceania'],
      dtype=object)

In [1821]:
index_to_drop = df_record[df_record['tournament_category'].isin(['Other/Regional', 'World Cup Qualifiers'])].index
df_record = df_record.drop(index=index_to_drop)

In [1822]:
df_record['tournament_category'].unique()


array(['Friendly', 'Africa', 'Europe', 'Asia', 'Americas', 'Oceania'],
      dtype=object)

In [1823]:
df_record = df_record.drop(['city', 'country', 'tournament', 'neutral'], axis=1)


In [1824]:
df_record

,date,home_team,away_team,home_score,away_score,tournament_category
17113,1990-01-12,Algeria,Mali,5,0,Friendly
17114,1990-01-14,Algeria,Cameroon,3,1,Friendly
17115,1990-01-17,Greece,Belgium,2,0,Friendly
17116,1990-01-17,Mexico,Argentina,2,0,Friendly
17117,1990-01-20,Malawi,Tanzania,2,2,Friendly
...,...,...,...,...,...,...
45759,2022-12-30,Iraq,Kuwait,1,0,Friendly
45760,2022-12-30,Oman,Syria,1,0,Friendly
45761,2022-12-30,United Arab Emirates,Lebanon,1,0,Friendly
45762,2022-12-30,Myanmar,Laos,2,2,Asia


---
matches 데이터

In [1825]:
df_fin

,Year,home_team,away_team,home_score,away_score,Score
0,2022,Argentina,France,3,3,(4) 3–3 (2)
1,2022,Croatia,Morocco,2,1,2–1
2,2022,France,Morocco,2,0,2–0
3,2022,Argentina,Croatia,3,0,3–0
4,2022,Morocco,Portugal,1,0,1–0
...,...,...,...,...,...,...
959,1930,Argentina,France,1,0,1–0
960,1930,Yugoslavia,Brazil,2,1,2–1
961,1930,Romania,Peru,3,1,3–1
962,1930,United States,Belgium,3,0,3–0


In [1826]:
df_fin["home_win"] = (df_fin["home_score"] > df_fin["away_score"]).map({True: 1, False: -1})

df_fin = df_fin.drop(['home_score', 'away_score'], axis=1)

In [1827]:
df_fin = df_fin.drop('Score', axis=1)

In [1828]:
df_fin

,Year,home_team,away_team,home_win
0,2022,Argentina,France,-1
1,2022,Croatia,Morocco,1
2,2022,France,Morocco,1
3,2022,Argentina,Croatia,1
4,2022,Morocco,Portugal,1
...,...,...,...,...
959,1930,Argentina,France,1
960,1930,Yugoslavia,Brazil,1
961,1930,Romania,Peru,1
962,1930,United States,Belgium,1


---

results 데이터

In [1829]:
df_record["home_win"] = (df_record["home_score"] > df_record["away_score"]).map({True: 1, False: -1})

df_count = df_record.drop(['home_score', 'away_score'], axis=1)

In [1830]:
df_count["year"] = pd.to_numeric(df_count["date"].str[:4]).astype(int)

In [1831]:
# 날짜 → 연도
df_count["year"] = df_count["date"].str[:4].astype(int)

# 연도 범위 매핑 함수
def map_year(y):
    if 1990 <= y <= 2001:
        return 2002
    elif 2001 < y <= 2005:
        return 2006
    elif 2005 < y <= 2009:
        return 2010
    elif 2009 < y <= 2013:
        return 2014
    elif 2013 < y <= 2017:
        return 2018
    else:
        return 2022

df_count["year"] = df_count["year"].apply(map_year)
df_count = df_count.drop('date', axis=1)

In [1832]:
df_count = df_count.drop('tournament_category', axis=1)

In [1833]:
df_count

,home_team,away_team,home_win,year
17113,Algeria,Mali,1,2002
17114,Algeria,Cameroon,1,2002
17115,Greece,Belgium,1,2002
17116,Mexico,Argentina,1,2002
17117,Malawi,Tanzania,-1,2002
...,...,...,...,...
45759,Iraq,Kuwait,1,2022
45760,Oman,Syria,1,2022
45761,United Arab Emirates,Lebanon,1,2022
45762,Myanmar,Laos,-1,2022


In [1834]:
df_count['home_team'].unique()

array(['Algeria', 'Greece', 'Mexico', 'Malawi', 'Eswatini', 'Botswana',
       'Kuwait', 'France', 'Liberia', 'Nigeria', 'United Arab Emirates',
       'Ivory Coast', 'Iran', 'Uganda', 'Togo', 'Bermuda', 'Egypt',
       'Iraq', 'Belgium', 'Gambia', 'Morocco', 'Netherlands', 'Spain',
       'United States', 'Cameroon', 'Senegal', 'Mozambique', 'Zambia',
       'Lesotho', 'Angola', 'Jamaica', 'Hungary', 'Basque Country',
       'Saint Kitts and Nevis', 'Northern Ireland', 'England',
       'German DR', 'Republic of Ireland', 'Luxembourg', 'Poland',
       'Russia', 'Scotland', 'Switzerland', 'Czechoslovakia', 'Austria',
       'Denmark', 'Germany', 'Israel', 'Sweden', 'Gabon', 'Guadeloupe',
       'Brazil', 'Cayman Islands', 'Andalusia', 'Barbados', 'Mauritania',
       'Zimbabwe', 'Wales', 'Romania', 'Turkey', 'Yugoslavia', 'Georgia',
       'Malta', 'Burkina Faso', 'Iceland', 'Italy', 'Liechtenstein',
       'Jordan', 'Tunisia', 'Norway', 'Namibia', 'Faroe Islands',
       'Réunion', '

In [1835]:
country_map = {
    'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',
    'IR Iran': 'Iran',
    "Côte d'Ivoire": 'Ivory Coast',
    'Türkiye': 'Turkey',
    'United States': 'United States',
    'Serbia and Montenegro': 'Serbia',
    'China PR': 'China',
    'German DR': 'Germany',
    'United States Virgin Islands': 'United States',
    'DR Congo': 'Congo',
    'Soviet Union': 'Russia',
    'Czechoslovakia': 'Czech Republic',
}

# before_record 기준 컬럼 변환
df_fin['home_team'] = df_fin['home_team'].map(lambda x: country_map.get(x, x))
df_fin['away_team'] = df_fin['away_team'].map(lambda x: country_map.get(x, x))

In [1836]:
df_fin['home_team'].unique()

array(['Argentina', 'Croatia', 'France', 'Morocco', 'England',
       'Netherlands', 'Portugal', 'Japan', 'Brazil', 'South Korea',
       'Ghana', 'Cameroon', 'Serbia', 'Canada', 'Costa Rica', 'Australia',
       'Tunisia', 'Saudi Arabia', 'Poland', 'Ecuador', 'Iran', 'Wales',
       'Belgium', 'Spain', 'Qatar', 'Switzerland', 'Uruguay', 'Germany',
       'Denmark', 'Mexico', 'Senegal', 'United States', 'Sweden',
       'Russia', 'Colombia', 'Panama', 'Iceland', 'Nigeria', 'Peru',
       'Egypt', 'Algeria', 'Bosnia and Herzegovina', 'Honduras', 'Italy',
       'Greece', 'Ivory Coast', 'Chile', 'Paraguay', 'North Korea',
       'Slovakia', 'Slovenia', 'South Africa', 'New Zealand', 'Ukraine',
       'Togo', 'Czech Republic', 'Angola', 'Trinidad and Tobago',
       'Turkey', 'China', 'Republic of Ireland', 'Romania', 'Scotland',
       'FR Yugoslavia', 'Jamaica', 'Bulgaria', 'Bolivia', 'Norway',
       'West Germany', 'Yugoslavia', 'Austria', 'United Arab Emirates',
       'Northern Irel

In [1837]:
# 컬럼 이름 통일
df_fin = df_fin.rename(columns={'Year':'year'})

# 두 데이터프레임 합치기
df_all = pd.concat([df_count, df_fin], ignore_index=True)

# 정렬
df_all = df_all.sort_values(by=['year', 'home_team', 'away_team']).reset_index(drop=True)
df_all

,home_team,away_team,home_win,year
0,Argentina,Chile,1,1930
1,Argentina,France,1,1930
2,Argentina,Mexico,1,1930
3,Argentina,United States,1,1930
4,Brazil,Bolivia,1,1930
...,...,...,...,...
19895,Zimbabwe,DR Congo,-1,2022
19896,Zimbabwe,DR Congo,-1,2022
19897,Zimbabwe,Guinea,1,2022
19898,Zimbabwe,Zambia,-1,2022


In [1838]:
# 홈팀 승리 횟수
df_win_count = df_all[df_all['home_win'] == 1].groupby(
    ['year', 'home_team', 'away_team']
).size().reset_index(name='win_count')

# 홈팀 vs 어웨이팀 총 경기 수
df_total_count = df_all.groupby(
    ['year', 'home_team', 'away_team']
).size().reset_index(name='total_count')

# 합치고 홈승 확률 계산
df_rate = pd.merge(df_total_count, df_win_count, on=['year', 'home_team', 'away_team'], how='left')

# NaN -> 0
df_rate['win_count'] = df_rate['win_count'].fillna(0)

# 홈승률 계산
df_rate['home_rate'] = df_rate['win_count'] / df_rate['total_count']

# 최종 선택
df_rate = df_rate[['year', 'home_team', 'away_team', 'home_rate']]

In [1839]:
df_rate.to_csv('df.csv')
df_rate

,year,home_team,away_team,home_rate
0,1930,Argentina,Chile,1.0
1,1930,Argentina,France,1.0
2,1930,Argentina,Mexico,1.0
3,1930,Argentina,United States,1.0
4,1930,Brazil,Bolivia,1.0
...,...,...,...,...
15725,2022,Zimbabwe,Comoros,1.0
15726,2022,Zimbabwe,Congo,1.0
15727,2022,Zimbabwe,DR Congo,0.0
15728,2022,Zimbabwe,Guinea,1.0


In [1840]:
df_diff['Home_Eng_Nation'] = df_diff['Home_Eng_Nation'].map(lambda x: country_map.get(x, x))
df_diff['Away_Eng_Nation'] = df_diff['Away_Eng_Nation'].map(lambda x: country_map.get(x, x))

In [1841]:
df_rate

,year,home_team,away_team,home_rate
0,1930,Argentina,Chile,1.0
1,1930,Argentina,France,1.0
2,1930,Argentina,Mexico,1.0
3,1930,Argentina,United States,1.0
4,1930,Brazil,Bolivia,1.0
...,...,...,...,...
15725,2022,Zimbabwe,Comoros,1.0
15726,2022,Zimbabwe,Congo,1.0
15727,2022,Zimbabwe,DR Congo,0.0
15728,2022,Zimbabwe,Guinea,1.0


In [1842]:
df_merged = df_diff.merge(
    df_rate,
    left_on=['Year', 'Home_Eng_Nation', 'Away_Eng_Nation'], 
    right_on=['year', 'home_team', 'away_team'],    
    how='left'  # 없는 경우 NaN
)

In [1843]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 317 entries, 0 to 316
Data columns (total 31 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Q_WR             317 non-null    float64
 1   Q_GR             317 non-null    float64
 2   F_Rank           317 non-null    float64
 3   F_Point          317 non-null    float64
 4   F_Rd             317 non-null    float64
 5   F_Pd             317 non-null    float64
 6   Avg_Apps         317 non-null    float64
 7   Avg_Age          317 non-null    float64
 8   Avg_Famous       317 non-null    int64  
 9   FS_0             317 non-null    float64
 10  FS_1             317 non-null    float64
 11  FS_2             317 non-null    float64
 12  FS_3             317 non-null    float64
 13  FS_4             317 non-null    float64
 14  FS_5             317 non-null    float64
 15  FS_6             317 non-null    float64
 16  FS_7             317 non-null    float64
 17  FS_8            

In [1844]:
df_merged = df_merged.drop(['Year', 'Home_Eng_Nation', 'Away_Eng_Nation', 'year', 'home_team', 'away_team'], axis=1)

In [1845]:
df_merged.to_csv('final.csv', index=False)

In [1846]:
df_merged.columns

Index(['Q_WR', 'Q_GR', 'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps',
       'Avg_Age', 'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5',
       'FS_6', 'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13',
       'Result', 'home_rate'],
      dtype='object')

In [1847]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=1
)
le = LabelEncoder()
y_encoded = le.fit_transform(df_merged['Result'])
results = cross_validate(model,df_merged.drop('Result',axis=1),y_encoded,cv=5,scoring = ['accuracy','precision_macro', 'recall_macro'])
for key in results:
    mean_val = np.mean(results[key])
    print(f"{key} 평균: {mean_val:.4f}")

fit_time 평균: 0.0479
score_time 평균: 0.0106
test_accuracy 평균: 0.9117
test_precision_macro 평균: 0.9133
test_recall_macro 평균: 0.9137
